# 05 — Variáveis derivadas (percentuais e densidade)

**Objetivo:** calcular a densidade demográfica (V2) e todos os percentuais (V4, V6, V8, V10…V20, V23, V29, V31,
V33…V55), para os três anos, a partir das contagens já compatibilizadas na malha de 2022. As fórmulas são lidas
diretamente da planilha corrigida gerada no notebook 01 — não são reescritas à mão.

**Entradas:**
- `output/04_deflacionado/<slug_cidade>_deflacionado.gpkg` (5 arquivos)
- `output/01_diagnostico/VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx` (coluna
  "FÓRMULA / DENOMINADOR (corrigida)")

**Saídas:**
- `output/05_derivadas/<slug_cidade>_completo.gpkg` (5 arquivos)

**Depende de:** `04_ipca_deflacionamento.ipynb`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd

from utils import CIDADES, ANOS, OUTPUT_DIR, parse_formula_percentual, calcula_percentual

IN_DIR = OUTPUT_DIR / "04_deflacionado"
OUT_DIR = OUTPUT_DIR / "05_derivadas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

XLSX_CORRIGIDO = OUTPUT_DIR / "01_diagnostico" / "VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 2. Leitura das fórmulas corrigidas

- A coluna `FÓRMULA / DENOMINADOR (corrigida)` traz duas formas: `V1 / ÁREA DO SETOR` (só para V2, tratada à
  parte, pois a área não é uma variável V*) e `Vx X 100 / Vy` ou `(Vx + Vz) X 100 / Vy` para todas as demais.
- Usa `parse_formula_percentual` (`src/utils.py`) para interpretar a segunda forma.

In [ ]:
dicionario = pd.read_excel(XLSX_CORRIGIDO, sheet_name="VARIÁVEIS FINAL", header=3)
dicionario = dicionario.dropna(subset=["VARIÁVEL"]).reset_index(drop=True)

col_formula = "FÓRMULA / DENOMINADOR (corrigida)"

formulas_percentual = {}
for _, linha in dicionario.iterrows():
    var = linha["VARIÁVEL"]
    formula = linha[col_formula]
    if not isinstance(formula, str) or not formula.strip() or var == "V2":
        continue
    formulas_percentual[var] = parse_formula_percentual(formula)

print(f"{len(formulas_percentual)} fórmulas de percentual carregadas.")
list(formulas_percentual.items())[:5]


## 3. Aplicação das fórmulas, por cidade e por ano

- Para cada ano (2000, 2010, 2022), aplica as fórmulas sobre as colunas daquele ano (ex.: `V4_2000` a partir de
  `V3_2000` e `V1_2000`).
- Premissa: denominador nulo ou zero resulta em percentual nulo (nunca zero), para não confundir "não se
  aplica" com "não tem".

In [ ]:
def calcula_densidade(gdf, ano):
    area_ha = gdf["area_m2"] / 10_000
    return (gdf[f"V1_{ano}"] / area_ha).where(area_ha > 0)


def aplica_derivadas(gdf):
    for ano in ANOS:
        gdf[f"V2_{ano}"] = calcula_densidade(gdf, ano)
        for var, formula in formulas_percentual.items():
            cols_num = [f"{v}_{ano}" for v in formula["numerador"]]
            col_denom = f"{formula['denominador']}_{ano}"
            if not all(c in gdf.columns for c in cols_num + [col_denom]):
                continue
            gdf[f"{var}_{ano}"] = calcula_percentual(gdf, cols_num, col_denom)
    return gdf


In [ ]:
for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    gdf = gpd.read_file(IN_DIR / f"{slug}_deflacionado.gpkg")
    gdf = aplica_derivadas(gdf)
    gdf.to_file(OUT_DIR / f"{slug}_completo.gpkg", driver="GPKG")
    print(f"{cidade}: {gdf.shape[1]} colunas -> {slug}_completo.gpkg")


## 4. Verificação: percentuais em [0, 100] e composições fecham em ~100%

Testa, para as 5 cidades e para 2022 (ano com cobertura completa de variáveis), se:
- todos os percentuais calculados ficam entre 0 e 100;
- a composição etária (V4+V6+V8), racial (V10+V12+V14+V16+V18) e por espécie de domicílio (V29+V33+V37) somam
  próximo de 100% onde os dados existem — conferência mais direta de que a correção das fórmulas do bloco de
  domicílios (notebook 01, seção 6) foi aplicada certo.

In [ ]:
colunas_percentual_2022 = [f"{v}_2022" for v in formulas_percentual] + ["V2_2022"]

linhas_composicao = []
fora_do_intervalo = {}
for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    gdf = gpd.read_file(OUT_DIR / f"{slug}_completo.gpkg")

    for col in colunas_percentual_2022:
        if col == "V2_2022":
            continue  # densidade não é percentual, não se aplica o teste [0,100]
        serie = gdf[col].dropna()
        n_fora = int(((serie < -0.01) | (serie > 100.01)).sum())
        if n_fora:
            fora_do_intervalo[(cidade, col)] = n_fora

    composicao_etaria = (gdf["V4_2022"] + gdf["V6_2022"] + gdf["V8_2022"]).dropna()
    composicao_racial = (gdf["V10_2022"] + gdf["V12_2022"] + gdf["V14_2022"] + gdf["V16_2022"] + gdf["V18_2022"]).dropna()
    composicao_domicilio = (gdf["V29_2022"] + gdf["V33_2022"] + gdf["V37_2022"]).dropna()

    for nome, serie in [
        ("etária (V4+V6+V8)", composicao_etaria),
        ("racial (V10+V12+V14+V16+V18)", composicao_racial),
        ("domicílio (V29+V33+V37)", composicao_domicilio),
    ]:
        linhas_composicao.append({
            "cidade": cidade, "composicao": nome,
            "media": serie.mean(), "min": serie.min(), "max": serie.max(),
        })

print("Colunas com valores fora de [0, 100]:", fora_do_intervalo or "nenhuma")

relatorio_composicao = pd.DataFrame(linhas_composicao)
relatorio_composicao


Médias próximas de 100 (com mínimo/máximo não muito distantes) confirmam que a leitura corrigida das fórmulas do
bloco de domicílios (correção da seção 6 do notebook 01) está correta — se a correção estivesse errada, a
composição por espécie de domicílio não fecharia perto de 100%.